# Download các thư viện cần thiết


In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [2]:
def read_parquet(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [3]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [4]:
def read_parquet_purchase(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [13]:
def split_and_save_parquet(df, num_files, output_dir):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.item_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Load các dataframe cần thiết

In [6]:
purchase_df = read_parquet_purchase("./Phase-2/preprocessed-dataset")
purchase_df

item_id,price,quantity,customer_id,created_date,channel,payment,location,discount,list_price,category_l2,discount_rate
str,"decimal[38,4]",i32,i32,date,str,str,i32,"decimal[38,4]","decimal[38,4]",str,"decimal[38,4]"
"""2596000000003""",296000.0000,1,7627794,2024-12-12,"""iOS""","""Tiền mặt""",578,149000.0000,445000.0000,"""Moony""",0.3348
"""1512000000004""",274349.0000,1,7367484,2024-12-12,"""SPE""","""Tiền mặt""",945,20651.0000,295000.0000,"""TPCN cho bé""",0.0700
"""5537000000015""",70562.0000,1,4795945,2024-12-12,"""SPE""","""Không xác định""",321,4438.0000,75000.0000,"""Snack ăn dặm""",0.0592
"""4644000000001""",58240.0000,4,7253028,2024-12-12,"""In-Store""","""Tiền mặt""",304,23040.0000,65000.0000,"""Caryn""",0.1040
"""2700000000002""",61750.0000,1,7681758,2024-12-12,"""SPE""","""Tiền mặt""",735,3250.0000,65000.0000,"""Khăn khô""",0.0500
…,…,…,…,…,…,…,…,…,…,…,…
"""0220000000004""",1050000.0000,1,7050031,2024-01-22,"""In-Store""","""Tiền mặt""",297,0.0000,1050000.0000,"""Chăm sóc gia đình""",0.0000
"""1236000000002""",170000.0000,1,7013488,2024-01-22,"""In-Store""","""Tiền mặt""",437,25000.0000,205000.0000,"""Bobby""",0.1707
"""4006000000010""",99000.0000,1,7124588,2024-01-22,"""In-Store""","""Tiền mặt""",730,20000.0000,119000.0000,"""Nón""",0.1681


In [16]:
item_df = read_parquet_item("./Phase-2/preprocessed-dataset")
item_df.head()

item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,item_type,description_new,gender_target_final,age_group_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Không xác định""","""Từ 9M"""
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé gái""","""Không xác định""","""Con Cưng""","""Bé Gái""","""Bộ quần áo""","""Không xác định""","""Bé Gái""","""Từ 36M"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Gặm nướu""","""Gặm nướu khác""","""- Chất liệu: Sản phẩm được làm…","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Không xác định""","""0-12M"""
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Sơ Sinh""","""﻿﻿Tã dán Merries size S 82 miế…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""[""Từ 4M"", ""3M-6M"", ""12-36M""]"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Tã Quần""","""﻿﻿﻿Bỉm tã quần Merries size M …","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""12-36M"""


In [15]:
item_df = item_df.drop(['age_group', 'age_group_from_desc_str', 'age_group_final_before'])
split_and_save_parquet(item_df, 1, "./Phase-2/preprocessed-dataset")

Đã lưu file: ./Phase-2/preprocessed-dataset/sale_pers.item_chunk_0.parquet


In [18]:
user_df = read_parquet("./Phase-2/preprocessed-dataset")
user_df.head()

customer_id,gender,location,province,membership,created_date,last_sync_date,location_name,install_app,install_datetime,user_age_days,days_since_install,days_since_last_sync
i32,str,i32,str,str,date,date,str,str,date,f64,f64,f64
1800754,"""female""",169,"""Hồ Chí Minh""","""Standard""",2018-09-08,2025-07-16,"""HCM - 178 Đỗ Xuân Hợp""","""In-Store""",2018-09-08,2579.16419,2579.917091,76.419849
1800755,"""female""",822,"""Lâm Đồng""","""Standard""",2018-09-08,2025-07-16,"""LDO - 4A - 4C Hải Thượng""","""In-Store""",2018-09-08,2579.1638,2579.917091,76.419849
1619184,"""male""",980,"""Cần Thơ""","""Standard""",2018-06-09,null,"""CTO - Hà Huy Giáp""","""In-Store""",2018-06-09,2670.237216,2670.917091,null
1800766,"""female""",591,"""Đồng Nai""","""Standard""",2018-09-08,2025-07-16,"""DON - 665 Quốc lộ 20""","""In-Store""",2018-09-08,2579.161078,2579.917091,76.419849
1800777,"""female""",971,"""Hồ Chí Minh""","""Standard""",2018-09-08,2025-07-16,"""HCM - 140A Hoàng Hoa Thám""","""In-Store""",2018-09-08,2579.157904,2579.917091,76.419849


# Task A: Hãy thống kê những sản phẩm hay mua chung và số lần mua chung: item 1 | item 2 | #cooc

Chuyển dữ liệu của trường `created_date` sang dạng `timestamp` để việc xử lý trở nên dễ dàng hơn 

In [9]:
import polars as pl
from itertools import combinations
from collections import Counter

# --- Giả sử df_purchase có cột: customer_id, invoice_id, item_id ---
# df_purchase = pl.read_parquet("purchases.parquet")

# 1️⃣ Gom các sản phẩm trong cùng một hóa đơn
df_grouped = (
    purchase_df.group_by(["customer_id", "created_date"])
    .agg(pl.col("item_id").unique().alias("items"))
)

# 2️⃣ Đếm số lần xuất hiện cặp sản phẩm
cooc_counter = Counter()
for items in df_grouped["items"]:
    if len(items) > 1:
        for pair in combinations(sorted(items), 2):
            cooc_counter[pair] += 1

# 3️⃣ Kết quả
cooc_count = pl.DataFrame({
    "item_1": [i1 for (i1, i2) in cooc_counter.keys()],
    "item_2": [i2 for (i1, i2) in cooc_counter.keys()],
    "cooc_count": list(cooc_counter.values())
}).sort("cooc_count", descending=True)

print(cooc_count.head(10))

shape: (10, 3)
┌───────────────┬───────────────┬────────────┐
│ item_1        ┆ item_2        ┆ cooc_count │
│ ---           ┆ ---           ┆ ---        │
│ str           ┆ str           ┆ i64        │
╞═══════════════╪═══════════════╪════════════╡
│ 2803000000011 ┆ 2803000000013 ┆ 78235      │
│ 2803000000012 ┆ 2803000000013 ┆ 53478      │
│ 2803000000011 ┆ 2803000000012 ┆ 52843      │
│ 2803000000010 ┆ 2803000000012 ┆ 34204      │
│ 2803000000010 ┆ 2803000000013 ┆ 32656      │
│ 1371000000001 ┆ 1371000000002 ┆ 29677      │
│ 3880000000001 ┆ 3880000000002 ┆ 27793      │
│ 2803000000010 ┆ 2803000000011 ┆ 26985      │
│ 1371000000003 ┆ 1371000000006 ┆ 26776      │
│ 0029250010001 ┆ 0029250010003 ┆ 26509      │
└───────────────┴───────────────┴────────────┘


In [10]:
cooc_count.write_parquet("./data/history-chunk/cooc_count.parquet")

In [11]:
pairs = (
    pl.concat([
        cooc_count.select(
            pl.col("item_1").alias("item_id"),
            pl.col("item_2").alias("co_item"),
            pl.col("cooc_count")
        ),
        cooc_count.select(
            pl.col("item_2").alias("item_id"),
            pl.col("item_1").alias("co_item"),
            pl.col("cooc_count")
        )
    ])
)

# 🧠 Lấy top 5 co_item có cooc_count cao nhất cho từng item_id
top_co_items = (
    pairs.sort(["item_id", "cooc_count"], descending=[False, True])
         .group_by("item_id")
         .agg(pl.col("co_item").head(10).alias("top10_co_items"))
)

# 🔗 Gộp vào item_df
item_df = item_df.join(top_co_items, on="item_id", how="left")

# ✅ Kết quả
print(item_df.select(["item_id", "top10_co_items"]).head())


shape: (5, 2)
┌───────────────┬─────────────────────────────────┐
│ item_id       ┆ top10_co_items                  │
│ ---           ┆ ---                             │
│ str           ┆ list[str]                       │
╞═══════════════╪═════════════════════════════════╡
│ 0502020000004 ┆ ["2707000000001", "00070900003… │
│ 0010290040150 ┆ null                            │
│ 0008010000015 ┆ ["0008170000235", "22630000000… │
│ 0020010000094 ┆ ["5950000000001", "15120000000… │
│ 0020010000098 ┆ ["1512000000004", "00200200001… │
└───────────────┴─────────────────────────────────┘


In [12]:
target_id = "0502020000004"

top10_list = (
    item_df
    .filter(pl.col("item_id") == target_id)
    .select("top10_co_items")
    .item()   # chuyển giá trị trong cột thành object Python
)

print(top10_list)

shape: (10,)
Series: '' [str]
[
	"2707000000001"
	"0007090000357"
	"0006040000432"
	"5358000000001"
	"4553000000003"
	"2752000000003"
	"6072000000003"
	"2766000000005"
	"6548000000002"
	"0020020000264"
]


In [ ]:
target_id = "0502020000004"

# 1️⃣ Lấy danh sách top 10 sản phẩm mua chung
top10_list = (
    item_df
    .filter(pl.col("item_id") == target_id)
    .select("top10_co_items")
    .item()  # chuyển list từ cột sang object Python
)

# 2️⃣ Lọc item_df để lấy thông tin chi tiết của các sản phẩm này
top10_info = (
    item_df
    .filter(pl.col("item_id").is_in(top10_list))
    .select(["item_id", "description"])
)

# 3️⃣ In kết quả
print(top10_info)

# Task B. Hãy dự đoán tuổi của em bé dựa trên:
- Thông tin "age_group" của bảng item: Ngày đầu tiên mua

- Thông tin sữa có chữ "Step 1": Ngày đầu tiên mua

- Thông tin sữa có chữ "Mom": Ngày cuối cùng mua

Tạo bảng dự đoán: | customer_id | first_date_buy_step 1 | age_by_step1 | first_date_buy_age_group_0-3M | age_by_age_group | last_date_buy_milk4mom | age_by_milk4mom.

In [26]:

# 1️⃣ Lấy danh sách item_id
step1_items = item_df.filter(
    pl.col("description").str.contains("(?i)Step 1") |
    pl.col("description_new").str.contains("(?i)Step 1")
)["item_id"]

mom_items = item_df.filter(
    pl.col("description").str.contains("(?i)Mom") |
    pl.col("description_new").str.contains("(?i)Mom")
)["item_id"]

age_group_items = item_df.filter(pl.col("age_group") == "0-3M")["item_id"]

# 2️⃣ Nhóm theo từng điều kiện
step1_df = (
    purchase_df.filter(pl.col("item_id").is_in(step1_items))
    .group_by("customer_id")
    .agg(pl.col("created_date").min().alias("first_date_buy_step1"))
)

age_group_df = (
    purchase_df.filter(pl.col("item_id").is_in(age_group_items))
    .group_by("customer_id")
    .agg(pl.col("created_date").min().alias("first_date_buy_age_group_0_3M"))
)

mom_df = (
    purchase_df.filter(pl.col("item_id").is_in(mom_items))
    .group_by("customer_id")
    .agg(pl.col("created_date").max().alias("last_date_buy_milk4mom"))
)

# 3️⃣ Join từng bước và xóa cột dư
pred_df = step1_df.join(age_group_df, on="customer_id", how="outer")

# Nếu join tạo ra customer_id_right → drop
if "customer_id_right" in pred_df.columns:
    pred_df = pred_df.drop("customer_id_right")

pred_df = pred_df.join(mom_df, on="customer_id", how="outer")

if "customer_id_right" in pred_df.columns:
    pred_df = pred_df.drop("customer_id_right")

# 4️⃣ Tính tuổi (đơn vị: ngày)
pred_df = pred_df.with_columns([
    (pl.col("first_date_buy_step1") - pl.col("last_date_buy_milk4mom"))
        .dt.total_days()
        .alias("age_by_step1"),

    (pl.col("first_date_buy_age_group_0_3M") - pl.col("last_date_buy_milk4mom"))
        .dt.total_days()
        .alias("age_by_age_group"),

    pl.lit(0).alias("age_by_milk4mom")
])


# 5️⃣ Kết quả
print(pred_df)


/tmp/ipykernel_1210710/1210756496.py:16: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  purchase_df.filter(pl.col("item_id").is_in(step1_items))
/tmp/ipykernel_1210710/1210756496.py:22: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  purchase_df.filter(pl.col("item_id").is_in(age_group_items))


shape: (351_074, 7)
┌─────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ customer_id ┆ first_date_b ┆ first_date_ ┆ last_date_b ┆ age_by_step ┆ age_by_age_ ┆ age_by_milk │
│ ---         ┆ uy_step1     ┆ buy_age_gro ┆ uy_milk4mom ┆ 1           ┆ group       ┆ 4mom        │
│ i32         ┆ ---          ┆ up_0_3M     ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│             ┆ datetime[μs] ┆ ---         ┆ datetime[μs ┆ i64         ┆ i64         ┆ i32         │
│             ┆              ┆ datetime[μs ┆ ]           ┆             ┆             ┆             │
│             ┆              ┆ ]           ┆             ┆             ┆             ┆             │
╞═════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 6787588     ┆ 2024-05-19   ┆ null        ┆ 2024-11-24  ┆ -189        ┆ null        ┆ 0           │
│             ┆ 09:46:25.487 ┆             ┆ 10:00:44.61 ┆             

/tmp/ipykernel_1210710/1210756496.py:28: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  purchase_df.filter(pl.col("item_id").is_in(mom_items))
/tmp/ipykernel_1210710/1210756496.py:34: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  pred_df = step1_df.join(age_group_df, on="customer_id", how="outer")
/tmp/ipykernel_1210710/1210756496.py:40: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  pred_df = pred_df.join(mom_df, on="customer_id", how="outer")
